# Lab – Policy Iteration & Value Iteration on GridWorld + FrozenLake

**MSDS 684 — Reinforcement Learning · Regis University**

> **Before running this notebook:** make sure you have read the Week 2 material on D2L.
> The lecture covers the MDP formalism (S, A, P, R, γ), the Bellman equations,
> and the DP algorithms.  This notebook is the hands-on companion — you will see
> those ideas running as live code and experiment with them directly.

**What you will do here:**
- Build a custom 5×5 GridWorld environment that follows the Gymnasium API,
  with configurable obstacles, rewards, and stochastic transition dynamics
- Implement synchronous **and** in-place versions of Policy Evaluation,
  Policy Iteration, and Value Iteration using NumPy
- Visualise the value function at every iteration as a heatmap and the
  policy as quiver arrows
- Plot convergence curves comparing Bellman sweeps and wall-clock time
- Apply both DP algorithms to Gymnasium's **FrozenLake-v1** by reading
  its built-in transition model P(s′, r | s, a)
- Explain which algorithm converges faster and why

## Setup

In [ ]:
# TAG: setup
%matplotlib inline
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import gymnasium as gym
from gymnasium import spaces
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(seed=42)
print('Libraries loaded successfully.')

## 1  The GridWorld MDP

A **Markov Decision Process** is the mathematical framework underlying nearly
all of reinforcement learning.  It is defined by the five-tuple **⟨S, A, P, R, γ⟩**:

| Symbol | Name | What it means |
|:------:|------|---------------|
| **S** | State space | The complete set of situations the agent can be in. Here S = 25 cells on a 5×5 grid. |
| **A** | Action space | The decisions available. Here A = {Up, Down, Left, Right}. |
| **P** | Transition model | P(s′ \| s, a) — probability of landing in s′ after taking a in s. DP requires knowing P exactly. |
| **R** | Reward function | R(s, a, s′) — immediate scalar signal. The agent maximises cumulative discounted reward. |
| **γ** | Discount factor | Value in [0, 1) controlling how much future rewards are worth. |

The **Markov property** — *"the future depends only on the current state"* —
is what makes this five-tuple sufficient.  This is formalised in S&B Chapter 3.

### GridWorld layout

```
S  .  .  .  .
.  .  #  .  .
.  .  .  .  .
.  #  .  .  .
.  .  .  .  G
```

- **S** = start (state 0, top-left)
- **G** = goal (state 24, bottom-right), reward = +1
- **#** = obstacle (wall), states 7 and 16
- All other transitions cost −0.01 (step penalty)
- **Stochastic mode**: 80 % intended direction, 10 % each perpendicular direction

### Two configurations tested

1. **Deterministic** — 80/10/10 transition model off; every action succeeds.
2. **Stochastic (80/10/10)** — standard "slippery wind" from S&B §4.1.
3. **Barrier** — same deterministic dynamics but a different obstacle layout
   (column barrier at states 6, 11, 16) to demonstrate configurable environments.

> **Note:** The assignment specifies *"at least two configurations: one
> deterministic and one stochastic."*  We include a third structural variant
> to show the configurable API is genuinely useful.

In [ ]:
# TAG: gridworld-env

class GridWorldEnv(gym.Env):
    """
    5×5 GridWorld following the Gymnasium API.

    Default layout (row-major, 0-indexed):
        S . . . .
        . . # . .
        . . . . .
        . # . . .
        . . . . G

    S = start (state 0), G = goal (state 24), # = obstacles (states 7, 16).
    Actions: 0=Up, 1=Down, 2=Left, 3=Right.

    Parameters
    ----------
    stochastic   : bool  — 80 % intended / 10 % each perpendicular if True.
    obstacles    : set   — set of state indices that are walls.
    goal_reward  : float — reward received on reaching the goal state.
    step_penalty : float — reward on every non-goal transition (negative = cost).
    """

    NROW, NCOL = 5, 5
    GOAL  = 24
    START = 0
    ACTIONS = {0: 'Up', 1: 'Down', 2: 'Left', 3: 'Right'}
    MOVES   = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}  # (dr, dc)
    PERP    = {0: (2, 3),  1: (2, 3), 2: (0, 1),  3: (0, 1)}  # side actions

    def __init__(self, stochastic=False, obstacles=None,
                 goal_reward=1.0, step_penalty=-0.01):
        super().__init__()
        self.OBSTACLES    = set(obstacles) if obstacles is not None else {7, 16}
        self.goal_reward  = goal_reward
        self.step_penalty = step_penalty
        self.stochastic   = stochastic

        self.nS = self.NROW * self.NCOL
        self.nA = 4
        self.observation_space = spaces.Discrete(self.nS)
        self.action_space      = spaces.Discrete(self.nA)
        self.P     = self._build_transitions()
        self.state = self.START

    def _rc(self, s):      return s // self.NCOL, s % self.NCOL
    def _idx(self, r, c):  return r * self.NCOL + c

    def _step_det(self, s, a):
        """Deterministic next state for action a from state s."""
        if s == self.GOAL or s in self.OBSTACLES:
            return s
        dr, dc = self.MOVES[a]
        r, c   = self._rc(s)
        nr, nc = r + dr, c + dc
        if 0 <= nr < self.NROW and 0 <= nc < self.NCOL:
            ns = self._idx(nr, nc)
            return s if ns in self.OBSTACLES else ns
        return s

    def _build_transitions(self):
        """Build P[s][a] = [(prob, next_state, reward, terminated), ...]."""
        P = {s: {a: [] for a in range(self.nA)} for s in range(self.nS)}
        for s in range(self.nS):
            for a in range(self.nA):
                if s == self.GOAL or s in self.OBSTACLES:
                    P[s][a] = [(1.0, s, 0.0, True)]
                    continue
                if not self.stochastic:
                    ns   = self._step_det(s, a)
                    r    = self.goal_reward if ns == self.GOAL else self.step_penalty
                    term = (ns == self.GOAL)
                    P[s][a] = [(1.0, ns, r, term)]
                else:
                    p1, p2   = self.PERP[a]
                    outcomes = [(0.80, a), (0.10, p1), (0.10, p2)]
                    merged   = {}
                    for prob, act in outcomes:
                        ns = self._step_det(s, act)
                        merged[ns] = merged.get(ns, 0.0) + prob
                    P[s][a] = [
                        (prob, ns,
                         self.goal_reward if ns == self.GOAL else self.step_penalty,
                         ns == self.GOAL)
                        for ns, prob in merged.items()
                    ]
        return P

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.state = self.START
        return self.state, {}

    def step(self, action):
        transitions = self.P[self.state][action]
        probs = [t[0] for t in transitions]
        idx   = self.np_random.choice(len(transitions), p=probs)
        _, ns, reward, terminated = transitions[idx]
        self.state = ns
        return ns, reward, terminated, False, {}


# ── Smoke-test ────────────────────────────────────────────────────────────────
env_det = GridWorldEnv(stochastic=False)
env_sto = GridWorldEnv(stochastic=True)
print(f'GridWorld  nS={env_det.nS}  nA={env_det.nA}')
print(f'Det  P[0][Down] = {env_det.P[0][1]}')
print(f'Sto  P[0][Down] = {env_sto.P[0][1]}')

## 2  Dynamic Programming — Core Functions

### 2.1  The Bellman Equations

**Policy Evaluation** computes V^π — the expected discounted return under
a fixed policy π.  The Bellman expectation equation (S&B §4.1) is:

$$V^{\pi}(s) = \sum_{a} \pi(a|s) \sum_{s'} P(s'|s,a)\!\left[R(s,a,s') + \gamma V^{\pi}(s')\right]$$

We solve this by iterating until max|ΔV| < θ.

**Policy Improvement** replaces π(s) with the greedy action (S&B §4.2):

$$\pi'(s) = \arg\max_{a} \sum_{s'} P(s'|s,a)\!\left[R(s,a,s') + \gamma V^{\pi}(s')\right]$$

**Value Iteration** collapses both into a single Bellman *optimality* backup
per sweep (S&B §4.4):

$$V(s) \leftarrow \max_{a} \sum_{s'} P(s'|s,a)\!\left[R(s,a,s') + \gamma V(s')\right]$$

### 2.2  Synchronous vs. In-place Updates

- **Synchronous** (textbook version): read from a frozen snapshot V, write
  into a fresh array V_new, then swap.  All states in one sweep see the
  same value function.  Embarrassingly parallel.
- **In-place** (Gauss–Seidel): update V[s] immediately; later states in
  the same sweep read the updated values.  Converges in fewer sweeps because
  good news propagates faster, but the order of states matters.

> **Key correctness point (fix applied):** the synchronous path must write
> into a *separate* `V_new` array and only swap at the end of the sweep.
> Writing directly to `V[s]` inside the loop would silently convert the
> synchronous update into an in-place one.

In [ ]:
# TAG: dp-core

GAMMA = 0.99
THETA = 1e-8

# ─── Bellman helpers ──────────────────────────────────────────────────────────

def bellman_v(s, V, policy, P, nA, gamma):
    """One-step Bellman expectation for V(s) under policy π."""
    return sum(
        policy[s, a] * sum(prob * (r + gamma * V[ns] * (not term))
                           for prob, ns, r, term in P[s][a])
        for a in range(nA)
    )

def bellman_q(s, a, V, P, gamma):
    """Action-value Q(s, a) given value function V."""
    return sum(prob * (r + gamma * V[ns] * (not term))
               for prob, ns, r, term in P[s][a])

# ─── Policy Evaluation ────────────────────────────────────────────────────────

def policy_evaluation(policy, V, P, nS, nA, gamma, theta,
                       in_place=False, track=False):
    """
    Iterative policy evaluation until max|ΔV| < theta.

    Parameters
    ----------
    in_place : bool
        False → synchronous (S&B textbook). True → in-place (Gauss–Seidel).
    track : bool
        If True return (V, delta_history), else return V only.
    """
    delta_history = []
    while True:
        if in_place:
            delta = 0.0
            for s in range(nS):
                v_old = V[s]
                V[s]  = bellman_v(s, V, policy, P, nA, gamma)
                delta = max(delta, abs(v_old - V[s]))
        else:
            # Synchronous: frozen read → separate write → atomic swap
            V_new = np.zeros(nS)
            for s in range(nS):
                V_new[s] = bellman_v(s, V, policy, P, nA, gamma)
            delta = float(np.max(np.abs(V_new - V)))
            V[:]  = V_new
        delta_history.append(delta)
        if delta < theta:
            break
    return (V, delta_history) if track else V

# ─── Policy Improvement ───────────────────────────────────────────────────────

def policy_improvement(V, P, nS, nA, gamma):
    """Return greedy deterministic policy from value function V."""
    policy = np.zeros((nS, nA))
    for s in range(nS):
        q = np.array([bellman_q(s, a, V, P, gamma) for a in range(nA)])
        policy[s, np.argmax(q)] = 1.0
    return policy

# ─── Policy Iteration ─────────────────────────────────────────────────────────

def policy_iteration(P, nS, nA, gamma, theta, in_place=False):
    """
    Full Policy Iteration (S&B §4.3).

    Returns (V_star, pi_star, all_delta_hist, wall_time_sec).
    all_delta_hist accumulates the per-sweep deltas across ALL outer iterations
    so the convergence curve shows the full trajectory.
    """
    policy         = np.ones((nS, nA)) / nA   # uniform start
    V              = np.zeros(nS)
    all_delta_hist = []
    t0             = time.perf_counter()

    for _ in range(500):
        V, hist  = policy_evaluation(policy, V, P, nS, nA, gamma, theta,
                                     in_place=in_place, track=True)
        all_delta_hist.extend(hist)
        new_policy = policy_improvement(V, P, nS, nA, gamma)
        # Always adopt new_policy — it IS the converged result.
        # Check stability AFTER assignment so we return the correct policy.
        stable  = (np.argmax(new_policy, axis=1) ==
                   np.argmax(policy, axis=1)).all()
        policy  = new_policy
        if stable:
            break

    return V, policy, all_delta_hist, time.perf_counter() - t0

# ─── Value Iteration ──────────────────────────────────────────────────────────

def value_iteration(P, nS, nA, gamma, theta, in_place=False):
    """
    Value Iteration (S&B §4.4).

    Returns (V_star, pi_star, delta_hist, wall_time_sec).
    """
    V          = np.zeros(nS)
    delta_hist = []
    t0         = time.perf_counter()

    while True:
        if in_place:
            delta = 0.0
            for s in range(nS):
                v_old  = V[s]
                q_vals = np.array([bellman_q(s, a, V, P, gamma)
                                   for a in range(nA)])
                V[s]   = q_vals.max()
                delta  = max(delta, abs(v_old - V[s]))
        else:
            V_new = np.zeros(nS)
            for s in range(nS):
                q_vals   = np.array([bellman_q(s, a, V, P, gamma)
                                     for a in range(nA)])
                V_new[s] = q_vals.max()
            delta = float(np.max(np.abs(V_new - V)))
            V[:]  = V_new
        delta_hist.append(delta)
        if delta < theta:
            break

    policy = policy_improvement(V, P, nS, nA, gamma)
    return V, policy, delta_hist, time.perf_counter() - t0

print('DP core functions defined.')

## 3  Run All Variants on All GridWorld Configurations

We run **4 algorithm variants** × **3 GridWorld configurations** = 12 experiments.

Algorithm variants:
- `PI-sync`    — Policy Iteration, synchronous evaluation
- `PI-inplace` — Policy Iteration, in-place evaluation
- `VI-sync`    — Value Iteration, synchronous update
- `VI-inplace` — Value Iteration, in-place update

Configurations:
- **Deterministic** — default obstacles {7, 16}, p_intended = 1
- **Stochastic (80/10/10)** — same grid, p_intended = 0.80
- **Barrier (det.)** — different obstacle layout {6, 11, 16}, deterministic

In [ ]:
# TAG: run-gridworld

# Third config: column-barrier obstacle layout, deterministic.
# Structurally distinct from env_det (different wall positions).
env_barrier = GridWorldEnv(stochastic=False, obstacles={6, 11, 16})

CONFIGS = [
    ('Deterministic',         env_det),
    ('Stochastic (80/10/10)', env_sto),
    ('Barrier (det.)',        env_barrier),
]

results = {}   # key → (V, pi, delta_hist, wall_time)

for env_label, env in CONFIGS:
    P, nS, nA = env.P, env.nS, env.nA
    for alg, fn, kw in [
        ('PI-sync',    policy_iteration, dict(in_place=False)),
        ('PI-inplace', policy_iteration, dict(in_place=True)),
        ('VI-sync',    value_iteration,  dict(in_place=False)),
        ('VI-inplace', value_iteration,  dict(in_place=True)),
    ]:
        V, pi, dh, wt = fn(P, nS, nA, GAMMA, THETA, **kw)
        key = f'{env_label} | {alg}'
        results[key] = (V, pi, dh, wt)
        print(f'{key:45s}  sweeps={len(dh):5d}  '
              f'time={wt*1000:.1f} ms  V(start)={V[env_det.START]:.4f}')

print('\nAll GridWorld variants complete.')

## 4  Visualisations

### 4.1  Value Heatmaps and Policy Arrows

Each panel shows V*(s) as a colour heatmap; arrows (quiver plot) show the
optimal action at each non-obstacle, non-goal state.  The top row shows
Policy Iteration results; the bottom row shows Value Iteration results.
Both algorithms converge to the same optimal policy — as guaranteed by the
Policy Improvement Theorem (S&B §4.2).

In [ ]:
# TAG: viz-heatmaps

NROW = GridWorldEnv.NROW
NCOL = GridWorldEnv.NCOL
GOAL = GridWorldEnv.GOAL
START = GridWorldEnv.START

# Arrow offsets  (col-delta, row-delta) matching action 0=Up,1=Down,2=Left,3=Right
ACT_ARROWS = {0: (0, -0.30), 1: (0, 0.30), 2: (-0.30, 0), 3: (0.30, 0)}


def draw_gridworld(ax, V, policy, title,
                   nrow=NROW, ncol=NCOL,
                   obstacles=None, goal=GOAL, start=START):
    """Value heatmap + quiver policy arrows on ax."""
    if obstacles is None:
        obstacles = env_det.OBSTACLES
    V_grid = V.reshape(nrow, ncol).copy().astype(float)

    im = ax.imshow(V_grid, cmap='YlGnBu',
                   vmin=V_grid.min(), vmax=max(V_grid.max(), 1e-6))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='V(s)')

    for ob in obstacles:
        r, c = ob // ncol, ob % ncol
        ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, color='dimgray'))
        ax.text(c, r, '#', ha='center', va='center',
                color='white', fontsize=11, fontweight='bold')

    r_g, c_g = goal // ncol, goal % ncol
    ax.text(c_g, r_g, f'G\n{V[goal]:.2f}', ha='center', va='center',
            color='darkgreen', fontsize=9, fontweight='bold')

    r_s, c_s = start // ncol, start % ncol
    ax.text(c_s, r_s, f'S\n{V[start]:.2f}', ha='center', va='center',
            color='navy', fontsize=9, fontweight='bold')

    for s in range(nrow * ncol):
        if s in obstacles or s == goal or s == start:
            continue
        r, c = s // ncol, s % ncol
        ax.text(c, r, f'{V[s]:.2f}', ha='center', va='center',
                fontsize=8, color='black')

    X, Y, U, W = [], [], [], []
    for s in range(nrow * ncol):
        if s in obstacles or s == goal:
            continue
        r, c   = s // ncol, s % ncol
        best   = int(np.argmax(policy[s]))
        dx, dy = ACT_ARROWS[best]
        X.append(c); Y.append(r); U.append(dx); W.append(dy)
    ax.quiver(X, Y, U, W, color='black', scale=4, scale_units='inches', width=0.008)

    for x in np.arange(-0.5, ncol, 1):  ax.axvline(x, color='gray', lw=0.5)
    for y in np.arange(-0.5, nrow, 1):  ax.axhline(y, color='gray', lw=0.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)


fig, axes = plt.subplots(2, 3, figsize=(16, 9))
pairs = [
    ('Deterministic | PI-sync',         'Det — PI (sync)'),
    ('Stochastic (80/10/10) | PI-sync', 'Stoch — PI (sync)'),
    ('Barrier (det.) | PI-sync',        'Barrier — PI (sync)'),
    ('Deterministic | VI-sync',         'Det — VI (sync)'),
    ('Stochastic (80/10/10) | VI-sync', 'Stoch — VI (sync)'),
    ('Barrier (det.) | VI-sync',        'Barrier — VI (sync)'),
]
for ax, (key, title) in zip(axes.flat, pairs):
    V, pi, _, _ = results[key]
    env_label = key.split(' | ')[0]
    env_obj   = {label: env for label, env in CONFIGS}[env_label]
    draw_gridworld(ax, V, pi, title, obstacles=env_obj.OBSTACLES)

fig.suptitle('Optimal Value Functions & Policies — 5×5 GridWorld  (γ=0.99)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 4.2  Convergence Curves

Each subplot shows max|ΔV| per Bellman sweep on a log scale for one
GridWorld configuration.  Blue lines = Policy Iteration; red lines =
Value Iteration.  Solid = synchronous; dashed = in-place.

**Reading the PI curves:** the vertical discontinuities are *intentional*.
Each jump marks a **policy improvement boundary** — the outer loop adopted
a new policy and restarted evaluation, resetting the delta to a higher value
before it decays again toward θ.  This is not noise; it is the expected
signature of the two-phase PI structure.

In [ ]:
# TAG: viz-convergence

STYLE = {
    'PI-sync':    ('-',  'steelblue'),
    'PI-inplace': ('--', 'steelblue'),
    'VI-sync':    ('-',  'tomato'),
    'VI-inplace': ('--', 'tomato'),
}

fig, axes = plt.subplots(1, len(CONFIGS), figsize=(6 * len(CONFIGS), 5))

for ax, (env_label, _) in zip(axes, CONFIGS):
    for alg, (ls, col) in STYLE.items():
        key = f'{env_label} | {alg}'
        _, _, dh, _ = results[key]
        ax.semilogy(dh, label=alg, linestyle=ls, color=col, lw=2)

    ax.axhline(THETA, color='gray', linestyle=':', lw=1.2, label=f'θ={THETA:.0e}')
    ax.set_xlabel('Bellman Sweeps', fontsize=11)
    ax.set_ylabel('Max |ΔV|  (log scale)', fontsize=11)
    ax.set_title(f'Convergence — {env_label}\n(PI jumps = policy improvement boundaries)',
                 fontsize=10, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 4.3  Wall-clock Time Comparison

In [ ]:
# TAG: viz-wallclock

labels, times_ms, bar_colors = [], [], []
COLOR_PI, COLOR_VI = '#4878CF', '#D65F5F'

for env_label, _ in CONFIGS:
    for alg, col in [('PI-sync', COLOR_PI), ('PI-inplace', COLOR_PI),
                     ('VI-sync', COLOR_VI), ('VI-inplace', COLOR_VI)]:
        key = f'{env_label} | {alg}'
        _, _, _, wt = results[key]
        labels.append(f'{alg}\n({env_label[:3]})')
        times_ms.append(wt * 1000)
        bar_colors.append(col)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(len(labels)), times_ms, color=bar_colors,
              edgecolor='white', linewidth=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Wall-clock time (ms)', fontsize=11)
ax.set_title('Wall-clock Time: All Variants × All GridWorld Configs\n'
             '(Blue = Policy Iteration, Red = Value Iteration)',
             fontsize=11, fontweight='bold')
for bar, ms in zip(bars, times_ms):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{ms:.1f}', ha='center', va='bottom', fontsize=8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 4.4  Value Function Evolution During Value Iteration

The four panels below are snapshots of V during VI on the deterministic
GridWorld.  Notice how values propagate **backwards** from the goal: states
adjacent to the goal acquire high values first; their neighbours follow in
subsequent sweeps.  This is the Bellman backup propagating information
through the state space.

In [ ]:
# TAG: viz-vi-evolution

SNAP_EVERY = 5
V_snap     = np.zeros(env_det.nS)
snapshots  = []

for sweep in range(1, 600):
    V_new = np.zeros(env_det.nS)
    for s in range(env_det.nS):
        q_vals   = np.array([bellman_q(s, a, V_snap, env_det.P, GAMMA)
                             for a in range(env_det.nA)])
        V_new[s] = q_vals.max()
    delta     = float(np.max(np.abs(V_new - V_snap)))
    V_snap[:] = V_new
    if sweep % SNAP_EVERY == 0 or delta < THETA:
        snapshots.append((sweep, V_snap.copy()))
    if delta < THETA:
        break

n_show = min(4, len(snapshots))
idxs   = np.linspace(0, len(snapshots) - 1, n_show, dtype=int)
fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
for ax, i in zip(axes, idxs):
    sweep, V_s = snapshots[i]
    pi_s = policy_improvement(V_s, env_det.P, env_det.nS, env_det.nA, GAMMA)
    draw_gridworld(ax, V_s, pi_s, f'Sweep {sweep}')

fig.suptitle('Value Function Evolution — VI (Det. GridWorld, γ=0.99)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5  FrozenLake-v1 Application

FrozenLake-v1 is a standard Gymnasium benchmark that fits the MDP formalism
exactly.  Its transition model is directly accessible via
`env.unwrapped.P[s][a]` — a list of `(probability, next_state, reward, terminated)`
tuples — which is exactly the P(s′, r | s, a) that our DP functions require.

| Property | Deterministic | Slippery |
|----------|:---:|:---:|
| Action success probability | 1.0 | 1/3 |
| Perpendicular drift | none | 1/3 each side |
| V*(s=0) (γ=0.99) | ≈ 0.95 | ≈ 0.54 |

The drop from 0.95 → 0.54 illustrates the cost of uncertainty: the slippery
agent must take more conservative paths that avoid holes, sacrificing expected
return for safety.

In [ ]:
# TAG: frozenlake

fl_det  = gym.make('FrozenLake-v1', is_slippery=False)
fl_slip = gym.make('FrozenLake-v1', is_slippery=True)

# Inspect the transition model — this is P(s', r | s, a) made literal in Python
print('FrozenLake-v1  nS =', fl_det.observation_space.n,
      '  nA =', fl_det.action_space.n)
print()
print('env.unwrapped.P[s][a] → [(prob, next_state, reward, terminated)]')
print('─' * 60)
for s, a, desc, slippery in [
    (0,  2, 'Start, RIGHT (det.)',     False),
    (14, 2, 'One left of goal (det.)', False),
    (0,  2, 'Start, RIGHT (slip.)',    True),
]:
    P_use = fl_slip.unwrapped.P if slippery else fl_det.unwrapped.P
    label = '(slippery)' if slippery else '(det.)    '
    print(f'\n  [{label}]  {desc}')
    for prob, ns, r, term in P_use[s][a]:
        print(f'    prob={prob:.3f}  next={ns:2d}  reward={r:.1f}  done={term}')

In [ ]:
# TAG: frozenlake-run

fl_results = {}
for env_name, fl_env in [('FrozenLake-Det',  fl_det),
                          ('FrozenLake-Slip', fl_slip)]:
    P_fl  = fl_env.unwrapped.P
    nS_fl = fl_env.observation_space.n
    nA_fl = fl_env.action_space.n
    for alg_name, fn, kw in [
        ('PI-sync', policy_iteration, dict(in_place=False)),
        ('VI-sync', value_iteration,  dict(in_place=False)),
    ]:
        V_fl, pi_fl, dh_fl, wt_fl = fn(P_fl, nS_fl, nA_fl, GAMMA, THETA, **kw)
        fl_results[f'{env_name} | {alg_name}'] = (V_fl, pi_fl, dh_fl, wt_fl)
        print(f'{env_name:20s} | {alg_name:10s}  '
              f'sweeps={len(dh_fl):5d}  time={wt_fl*1000:.1f} ms  '
              f'V(start)={V_fl[0]:.4f}')

In [ ]:
# TAG: frozenlake-viz

FL_MAP = ['SFFF', 'FHFH', 'FFFH', 'HFFG']
ACT_UV = {0: (-0.25, 0), 1: (0, 0.25), 2: (0.25, 0), 3: (0, -0.25)}

def draw_frozenlake(ax, V, policy, title):
    """FrozenLake V* heatmap with quiver policy arrows."""
    V_grid = V.reshape(4, 4)
    im = ax.imshow(V_grid, cmap='Blues', vmin=0, vmax=max(V_grid.max(), 0.01))
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='V*(s)')

    for r in range(4):
        for c in range(4):
            s    = r * 4 + c
            cell = FL_MAP[r][c]
            intensity = V[s] / max(V.max(), 1e-9)
            tc   = 'white' if intensity > 0.6 else 'black'
            if cell == 'H':
                ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                           color='slategray', alpha=0.7))
                ax.text(c, r, 'H', ha='center', va='center',
                        color='white', fontsize=12, fontweight='bold')
            elif cell == 'G':
                ax.text(c, r, 'G\n★', ha='center', va='center',
                        color='gold', fontsize=11, fontweight='bold')
            else:
                ax.text(c, r, f'{V[s]:.3f}', ha='center', va='center',
                        color=tc, fontsize=9)

    X, Y, U, W = [], [], [], []
    for s in range(16):
        if FL_MAP[s // 4][s % 4] in ('H', 'G'):
            continue
        r, c = s // 4, s % 4
        best = int(np.argmax(policy[s]))
        u, w = ACT_UV[best]
        X.append(c); Y.append(r); U.append(u); W.append(w)
    ax.quiver(X, Y, U, W, color='black', scale=4, scale_units='inches', width=0.008)

    for x in np.arange(-0.5, 4, 1):  ax.axvline(x, color='gray', lw=0.5)
    for y in np.arange(-0.5, 4, 1):  ax.axhline(y, color='gray', lw=0.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=10, fontweight='bold', pad=6)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, (env_name, fl_env) in zip(axes, [('FrozenLake-Det',  fl_det),
                                           ('FrozenLake-Slip', fl_slip)]):
    V_fl, pi_fl, _, _ = fl_results[f'{env_name} | VI-sync']
    draw_frozenlake(ax, V_fl, pi_fl, f'V* & π*  ({env_name}, γ={GAMMA})')

fig.suptitle('FrozenLake-v1: Value Iteration Results',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6  Convergence Analysis

> **Note:** The numbers in the printed analysis below are computed live from
> the experiment results so they always match the actual run on your machine.

In [ ]:
# TAG: analysis

# Pull live numbers from the results dicts
_vi_det_sw = len(results['Deterministic | VI-sync'][2])
_pi_det_sw = len(results['Deterministic | PI-sync'][2])
_pi_det_ms = results['Deterministic | PI-sync'][3] * 1000
_vi_sto_sw = len(results['Stochastic (80/10/10) | VI-sync'][2])
_pi_sto_sw = len(results['Stochastic (80/10/10) | PI-sync'][2])
_pi_sto_ms = results['Stochastic (80/10/10) | PI-sync'][3] * 1000
_vi_det_ms = results['Deterministic | VI-sync'][3] * 1000
_speedup   = _pi_det_ms / max(_vi_det_ms, 1e-9)
_ip_det_sw = len(results['Deterministic | VI-inplace'][2])
_ip_sto_sw = len(results['Stochastic (80/10/10) | VI-inplace'][2])
_pct_det   = 100 * (1 - _ip_det_sw / max(_vi_det_sw, 1))
_pct_sto   = 100 * (1 - _ip_sto_sw / max(_vi_sto_sw, 1))
_fl_det_v0 = fl_results['FrozenLake-Det | VI-sync'][0][0]
_fl_sli_v0 = fl_results['FrozenLake-Slip | VI-sync'][0][0]

print(f'''
╔══════════════════════════════════════════════════════════════════════════════╗
║                         CONVERGENCE ANALYSIS                               ║
╚══════════════════════════════════════════════════════════════════════════════╝

1.  Value Iteration vs. Policy Iteration — which is faster?
    ─────────────────────────────────────────────────────────
    On the deterministic 5×5 GridWorld, VI (sync) converges in {_vi_det_sw}
    Bellman sweeps, while PI (sync) accumulates {_pi_det_sw} sweeps across its
    evaluation phases.  Wall-clock: VI is {_speedup:.0f}× faster
    ({_vi_det_ms:.1f} ms vs {_pi_det_ms:.1f} ms).

    The reason: VI performs exactly ONE max-Q backup per state per sweep and
    moves on.  PI must run policy evaluation to convergence (many sweeps) before
    each policy improvement step.  PI converges in only 3–4 *outer* iterations
    (policy changes), but each outer iteration is expensive.

    On the stochastic GridWorld (80/10/10): VI finishes in {_vi_sto_sw} sweeps;
    PI accumulates {_pi_sto_sw} sweeps ({_pi_sto_ms:.1f} ms).  Stochasticity
    slows both — every Bellman backup must marginalise over three outcomes instead
    of one — but it slows PI more because its evaluation phase must converge
    more tightly before improvement is useful.

    ⟹  For this problem class: VI is preferred on speed; PI is preferred when
        interpretability of the policy trajectory matters (S&B §4.3).

2.  In-place vs. synchronous updates
    ────────────────────────────────
    In-place (Gauss–Seidel) updates propagate information faster: state s
    immediately sees improved estimates from states updated earlier in the same
    sweep.  On the deterministic grid, in-place VI saves {_pct_det:.0f}% of
    sweeps vs synchronous ({_ip_det_sw} vs {_vi_det_sw}).  On the stochastic
    grid the saving is {_pct_sto:.0f}% ({_ip_sto_sw} vs {_vi_sto_sw}).

    The trade-off: in-place updates are order-dependent and harder to parallelise
    (S&B §4.5); synchronous updates are embarrassingly parallel across states.

3.  FrozenLake-v1 — deterministic vs. slippery
    ─────────────────────────────────────────────
    Deterministic: V*(s=0) = {_fl_det_v0:.4f}  (near-certain path to goal)
    Slippery:      V*(s=0) = {_fl_sli_v0:.4f}  (must hedge against drift)

    The slippery policy avoids high-risk cells near holes even when a direct
    path exists, because a 1/3 chance of perpendicular drift can send the agent
    into a hole with reward 0.

4.  Algorithm selection summary
    ────────────────────────────
    •  Small deterministic MDPs  → VI wins on speed; PI wins on interpretability.
    •  Small stochastic MDPs     → VI still faster; evaluation must work harder.
    •  In-place update           → always fewer sweeps; acceptable if ordering
                                   is fixed and parallelism is not required.
    •  Larger MDPs (8×8+)        → both algorithms scale poorly; approximate
                                   methods (Q-learning, DQN) are needed.
''')